In [0]:
from pyspark.sql import functions as F

In [0]:
df_silver = spark.table("us_economy_db.silver_bsky_sentiment")

In [0]:
df_enriched = (
    df_silver.withColumn("post_year", F.year(F.col("post_date")))
    .withColumn("post_month", F.month(F.col("post_date")))
    .withColumn("year_month", F.date_format(F.col("post_date"), "yyyy-MM"))
)

In [0]:
df_gold_daily = (
    df_enriched.groupBy("post_date", "post_hour", "search_topic")
    .agg(
        F.count("post_uri").alias("total_posts"),
        F.round(F.avg("sentiment_score"), 3).alias("avg_sentiment_score"),
        F.sum(F.when(F.col("sentiment_label") == "Positive", 1).otherwise(0)).alias("bullish_positive_posts"),
        F.sum(F.when(F.col("sentiment_label") == "Negative", 1).otherwise(0)).alias("bearish_negative_posts"),
        F.sum(F.when(F.col("sentiment_label") == "Neutral", 1).otherwise(0)).alias("neutral_posts"),
    )
    .withColumn(
        "bullish_ratio_pct",
        F.round((F.col("bullish_positive_posts") / F.col("total_posts")) * 100, 1),
    )
    .withColumn(
        "bearish_ratio_pct",
        F.round((F.col("bearish_negative_posts") / F.col("total_posts")) * 100, 1),
    )
    .withColumn("gold_updated_at", F.current_timestamp())
    .orderBy(F.desc("post_date"), F.desc("post_hour"))
)

df_gold_daily.write.format("delta").mode("overwrite").saveAsTable(
    "us_economy_db.gold_market_sentiment_summary"
)

df_gold_monthly = (
    df_enriched.groupBy("year_month", "search_topic", "sentiment_label")
    .agg(
        F.count("post_uri").alias("post_count"),
        F.round(F.avg("sentiment_score"), 3).alias("avg_sentiment_score"),
    )
    .orderBy("year_month", "search_topic")
)

df_gold_monthly.write.format("delta").mode("overwrite").saveAsTable(
    "us_economy_db.gold_monthly_trends"
)

In [0]:
df_gold_authors = (
    df_enriched.groupBy("author_handle", "search_topic")
    .agg(
        F.count("post_uri").alias("total_posts"),
        F.round(F.avg("sentiment_score"), 3).alias("avg_sentiment"),
        F.sum(F.when(F.col("sentiment_label") == "Positive", 1).otherwise(0)).alias("positive_posts"),
        F.sum(F.when(F.col("sentiment_label") == "Negative", 1).otherwise(0)).alias("negative_posts"),
    )
    .orderBy(F.desc("total_posts"))
)

df_gold_authors.write.format("delta").mode("overwrite").saveAsTable(
    "us_economy_db.gold_top_market_voices"
)

print("تم بناء جميع جداول طبقة Gold بنجاح في المستودع!")
display(spark.table("us_economy_db.gold_market_sentiment_summary"))